# 02 · Silver · `atendimentos`

Lê `v_energy.bronze.atendimentos`, abre o histórico aninhado e grava dados confiáveis para a Gold. O notebook consulta o schema real antes de transformar qualquer coluna; se o dataset usar um nome diferente do esperado, a execução para com uma mensagem explícita.

**Grãos:** `silver.atendimentos`: uma linha por atendimento; `silver.tratativas_atendimento`: uma linha por evento do histórico de um atendimento.

**Fontes:** Bronze preservada pela ingestão. A lista de aliases abaixo serve somente para acomodar variações de nomenclatura que serão confirmadas pela inspeção do schema. Nenhuma coluna é criada silenciosamente.

## 1. Parâmetros do ambiente

In [ ]:
catalog = "v_energy"
gold_schema = f"{catalog}.gold"
bronze_schema = f"{catalog}.bronze"
silver_schema = f"{catalog}.silver"
landing_path = f"/Volumes/{catalog}/bronze/landing"
tabela_bronze = f"{bronze_schema}.atendimentos"
tabela_atendimentos = f"{silver_schema}.atendimentos"
tabela_tratativas = f"{silver_schema}.tratativas_atendimento"
from datetime import datetime
from pyspark.sql import functions as F, Row
from pyspark.sql.types import ArrayType, StructType
print(f"origem: {tabela_bronze}")
print(f"destinos: {tabela_atendimentos}, {tabela_tratativas}")

## 2. Schema real e funções auxiliares

A inspeção vem antes das transformações para não assumir colunas que não existem.

In [ ]:
bronze = spark.table(tabela_bronze)
bronze.printSchema()
print(f"Linhas na Bronze: {bronze.count():,}".replace(",", "."))
display(bronze.limit(5))

def coluna_existente(df, aliases, obrigatoria=True):
    encontradas = [c for c in aliases if c in df.columns]
    if encontradas:
        return encontradas[0]
    if obrigatoria:
        raise ValueError(f"Nenhuma coluna encontrada para {aliases}. Colunas reais: {df.columns}")
    return None

id_atendimento_col = coluna_existente(bronze, ["id_atendimento", "protocolo", "atendimento_id", "id_chamado"])
id_uc_col = coluna_existente(bronze, ["id_uc", "uc", "unidade_consumidora"])
historico_col = coluna_existente(bronze, ["historico", "histórico"])
data_abertura_col = coluna_existente(bronze, ["data_abertura", "data_atendimento", "dt_abertura"], False)
data_fechamento_col = coluna_existente(bronze, ["data_fechamento", "dt_fechamento"], False)
nota_col = coluna_existente(bronze, ["nota_satisfacao", "satisfacao", "nota"], False)
print({"id_atendimento": id_atendimento_col, "id_uc": id_uc_col, "historico": historico_col, "data_abertura": data_abertura_col, "data_fechamento": data_fechamento_col, "nota_satisfacao": nota_col})

## 3. Cabeçalhos repetidos e cópias

**Decisão:** remover cabeçalhos que tenham sido lidos como dados e deduplicar pelo identificador do atendimento. O filtro é feito antes da deduplicação, seguindo o padrão já usado em `faturas` e `leituras`. As contagens reais serão impressas pelo notebook e registradas no log após a execução.

In [ ]:
n0 = bronze.count()
sem_cabecalho = bronze.filter(F.col(id_atendimento_col) != F.lit(id_atendimento_col))
n1 = sem_cabecalho.count()
print(f"Cabeçalhos removidos: {n0 - n1} | antes: {n0} | depois: {n1}")
sem_copias = sem_cabecalho.dropDuplicates([id_atendimento_col])
n2 = sem_copias.count()
print(f"Cópias removidas: {n1 - n2} | antes: {n1} | depois: {n2}")

## 4. Integridade com unidades consumidoras

**Decisão:** remover atendimentos sem UC ou com UC inexistente. Para a visão 360, um atendimento sem unidade não pode ser relacionado a cadastro, medição ou faturamento. A Bronze preserva os registros rejeitados.

In [ ]:
ids_uc = spark.table(f"{bronze_schema}.unidades_consumidoras").select("id_uc").distinct()
validos = (sem_copias
    .filter(F.col(id_uc_col).isNotNull())
    .join(ids_uc, sem_copias[id_uc_col] == ids_uc.id_uc, "left_semi")
)
n3 = validos.count()
print(f"Atendimentos sem UC ou UC órfã removidos: {n2 - n3} | antes: {n2} | depois: {n3}")

## 5. Padronização e tipagem

**Decisão:** preservar os campos originais relevantes e criar versões tipadas/padronizadas. Datas são convertidas com os formatos observados no case; a nota aceita vírgula decimal. Nulos de negócio permanecem nulos.

In [ ]:
def texto_padronizado(df, origem, destino):
    return df.withColumn(destino, F.upper(F.trim(F.col(origem))))

def data_flexivel(coluna):
    c = F.col(coluna).cast("string")
    c = F.when(F.trim(c) == "9999-12-31", F.lit(None)).otherwise(c)
    return F.coalesce(
        F.to_timestamp(c, "yyyy-MM-dd HH:mm:ss"),
        F.to_timestamp(c, "yyyy-MM-dd"),
        F.to_timestamp(c, "dd/MM/yyyy HH:mm:ss"),
        F.to_timestamp(c, "dd/MM/yyyy"),
        F.to_timestamp(c, "dd.MM.yyyy"),
        F.to_timestamp(c, "MM-dd-yyyy"),
        F.to_timestamp(c, "MM-dd-yyyy HH:mm:ss")
    )

at = validos.withColumnRenamed(id_atendimento_col, "id_atendimento").withColumnRenamed(id_uc_col, "id_uc")
for c in [c for c in ["canal", "assunto", "motivo", "status"] if c in at.columns]:
    at = at.withColumn(c, F.upper(F.trim(F.col(c))))
if data_abertura_col:
    at = at.withColumn("data_abertura", data_flexivel(data_abertura_col))
if data_fechamento_col:
    at = at.withColumn("data_fechamento", data_flexivel(data_fechamento_col))
if nota_col:
    at = at.withColumn("nota_satisfacao_original", F.col(nota_col))
    at = at.withColumn("nota_satisfacao", F.regexp_replace(F.trim(F.col(nota_col).cast("string")), ",", ".").cast("decimal(5,2)"))
at = at.withColumn("atendimento_em_aberto", F.when(F.col("data_fechamento").isNull(), True).otherwise(False) if "data_fechamento" in at.columns else F.lit(None).cast("boolean"))
atendimentos_silver = at.drop(historico_col) if historico_col in at.columns else at
for coluna_bruta, coluna_final in [(data_abertura_col, "data_abertura"), (data_fechamento_col, "data_fechamento"), (nota_col, "nota_satisfacao")]:
    if coluna_bruta and coluna_bruta != coluna_final and coluna_bruta in atendimentos_silver.columns:
        atendimentos_silver = atendimentos_silver.drop(coluna_bruta)
atendimentos_silver.printSchema()

## 6. Abertura do histórico

**Decisão:** manter o atendimento em uma tabela pai e abrir o array `historico` em `silver.tratativas_atendimento`. Isso evita arrays na Silver e conserva a granularidade de uma linha por tratativa.

In [ ]:
historico_tipo = dict(bronze.dtypes).get(historico_col)
if not isinstance(bronze.schema[bronze.columns.index(historico_col)].dataType, ArrayType):
    raise TypeError(f"A coluna {historico_col} não é array; schema encontrado: {historico_tipo}. Não vou achatar uma estrutura diferente sem revisão.")
tratativas = (
    validos.select(F.col(id_atendimento_col).alias("id_atendimento"), F.posexplode_outer(F.col(historico_col)).alias("ordem_tratativa", "tratativa"))
    .filter(F.col("tratativa").isNotNull())
)
if isinstance(bronze.schema[bronze.columns.index(historico_col)].dataType.elementType, StructType):
    tratativas_silver = tratativas.select("id_atendimento", "ordem_tratativa", "tratativa.*")
else:
    tratativas_silver = tratativas.withColumnRenamed("tratativa", "tratativa_valor")
tratativas_silver = tratativas_silver.dropDuplicates(["id_atendimento", "ordem_tratativa"])
tratativas_silver.printSchema()

## 7. Checagens de qualidade e gravação

As checagens são gravadas mesmo quando falham; as tabelas só são gravadas se todas passarem.

In [ ]:
dq_results = []
def dq_check(table_name, check_name, df, condition):
    total = df.count(); failed = df.filter(~condition).count(); passed = failed == 0
    dq_results.append(Row(table_name=table_name, check_name=check_name, total_rows=total, failed_rows=failed, passed=passed, checked_at=datetime.now()))
    print(f"[{'PASS' if passed else 'FAIL'}] {table_name} | {check_name} | {failed}/{total}")
def dq_check_unique(table_name, check_name, df, keys):
    total = df.count(); failed = df.groupBy(*keys).count().filter("count > 1").count(); passed = failed == 0
    dq_results.append(Row(table_name=table_name, check_name=check_name, total_rows=total, failed_rows=failed, passed=passed, checked_at=datetime.now()))
    print(f"[{'PASS' if passed else 'FAIL'}] {table_name} | {check_name} | {failed} chaves duplicadas")

dq_check_unique("silver.atendimentos", "id_atendimento único", atendimentos_silver, ["id_atendimento"])
dq_check("silver.atendimentos", "id_atendimento não nulo", atendimentos_silver, F.col("id_atendimento").isNotNull())
dq_check("silver.atendimentos", "id_uc não nulo", atendimentos_silver, F.col("id_uc").isNotNull())
dq_check_unique("silver.tratativas_atendimento", "atendimento e ordem únicos", tratativas_silver, ["id_atendimento", "ordem_tratativa"])
dq_check("silver.tratativas_atendimento", "atendimento da tratativa existe", tratativas_silver, F.col("id_atendimento").isNotNull())

dq_df = spark.createDataFrame(dq_results)
dq_df.write.format("delta").mode("append").saveAsTable(f"{silver_schema}.dq_log")
falhas = [r for r in dq_results if not r.passed]
if falhas:
    raise Exception(f"Silver não gravada: {len(falhas)} checagem(ns) falharam. Consulte silver.dq_log.")
atendimentos_silver.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(tabela_atendimentos)
tratativas_silver.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(tabela_tratativas)
print(f"Gravadas: {tabela_atendimentos} e {tabela_tratativas}")

## 8. Conferência Bronze × Silver

A amostra permite verificar a origem e o resultado tratado lado a lado.

In [ ]:
display(bronze.select(id_atendimento_col, id_uc_col, historico_col).limit(5))
display(spark.table(tabela_atendimentos).limit(5))
display(spark.table(tabela_tratativas).limit(10))